# Coursework: YOLO Teacher → Pruned Student → Logits Distillation

This notebook follows the style of Tutorial 4 (pruning) and implements a first-pass YOLO distillation workflow:

1. Load a YOLO teacher from Ultralytics checkpoint.
2. Convert to MASE graph and run unstructured pruning.
3. Treat the pruned model as the student.
4. Re-train the student with logits distillation from the teacher.

For a fast smoke test, this notebook uses synthetic images instead of a full detection dataset.

In [2]:
teacher_checkpoint = "yolov8n.pt"
device = "cuda"
image_size = 320
batch_size = 2
prune_sparsity = 0.3
kd_alpha = 0.8
kd_temperature = 2.0
kd_steps = 5

## Imports and setup

In [3]:
import sys
from pathlib import Path

import torch
from ultralytics import YOLO

def find_repo_root(start: Path) -> Path:
    """Find repository root by locating pyproject.toml and src directory."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/")

repo_root = find_repo_root(Path.cwd().resolve())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from chop import MaseGraph
import chop.passes as passes
from chop.models.yolo import get_yolo_detection_model

from mase_kd.core.losses import DistillationLossConfig
from mase_kd.vision.yolo_kd import YOLOLogitsDistiller

print(f"Repo root: {repo_root}")
print(f"Using src path: {src_path}")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.11/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


Repo root: /workspace/mase-kd
Using src path: /workspace/mase-kd/src


In [4]:
if device == "cuda" and not torch.cuda.is_available():
    device = "cpu"

print(f"Using device: {device}")
torch.set_grad_enabled(True)

Using device: cuda


## Load teacher model from Ultralytics

In [5]:
ultra_teacher = YOLO(teacher_checkpoint)
teacher_model = get_yolo_detection_model(teacher_checkpoint)
teacher_model = teacher_model.to(device)
teacher_model.eval()

print(f"Loaded teacher checkpoint: {teacher_checkpoint}")


                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128

## Build a MASE graph and prune to create the student

In [6]:
trace_input = torch.randn(1, 3, image_size, image_size)

# Build a fresh train-mode YOLO model for tracing/pruning.
# Training mode avoids postprocess/NMS control flow in eval forward.
student_seed_model = get_yolo_detection_model(teacher_checkpoint)
student_seed_model.train()

mg = MaseGraph(student_seed_model, cf_args={"x": trace_input})
mg, _ = passes.init_metadata_analysis_pass(mg)

# Build dummy inputs keyed by actual FX placeholder names (e.g. x, x_1).
placeholder_names = [
    node.name for node in mg.fx_graph.nodes if node.op == "placeholder"
 ]
dummy_in = {name: trace_input for name in placeholder_names}

mg, _ = passes.add_common_metadata_analysis_pass(
    mg,
    pass_args={
        "dummy_in": dummy_in,
        "add_value": True,
    },
)

pruning_config = {
    "weight": {
        "sparsity": prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
        "granularity": "elementwise",
    },
    "activation": {
        "sparsity": prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
        "granularity": "elementwise",
    },
}

mg, _ = passes.prune_transform_pass(mg, pass_args=pruning_config)
student_model = mg.model.to(device)

print(f"Metadata placeholders: {placeholder_names}")
print("Pruning complete; using pruned model as student.")


                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128

/usr/local/lib/python3.11/dist-packages/torch/fx/_symbolic_trace.py:906: UserWarning: Was not able to add assertion to guarantee correct input x to specialized function. It is up to the user to make sure that your inputs match the inputs you specialized the function with.
  warnings.warn(
INFO     Pruning module: model_0_conv
INFO     Pruning module: model_1_conv
INFO     Pruning module: model_2_cv1_conv
INFO     Pruning module: model_2_m_0_cv1_conv
INFO     Pruning module: model_2_m_0_cv2_conv
INFO     Pruning module: model_2_cv2_conv
INFO     Pruning module: model_3_conv
INFO     Pruning module: model_4_cv1_conv
INFO     Pruning module: model_4_m_0_cv1_conv
INFO     Pruning module: model_4_m_0_cv2_conv
INFO     Pruning module: model_4_m_1_cv1_conv
INFO     Pruning module: model_4_m_1_cv2_conv
INFO     Pruning module: model_4_cv2_conv
INFO     Pruning module: model_5_conv
INFO     Pruning module: model_6_cv1_conv
INFO     Pruning module: model_6_m_0_cv1_conv
INFO     Pruning module: m

Metadata placeholders: ['x_1']
Pruning complete; using pruned model as student.


## Distill teacher into pruned student

In [ ]:
kd_config = DistillationLossConfig(alpha=kd_alpha, temperature=kd_temperature)
distiller = YOLOLogitsDistiller(
    teacher=teacher_model,
    student=student_model,
    kd_config=kd_config,
    device=device,
)

optimizer = torch.optim.Adam(student_model.parameters(), lr=1e-4)

loss_history = []
for step in range(1, kd_steps + 1):
    images = torch.randn(batch_size, 3, image_size, image_size)
    batch = {"images": images}
    output = distiller.train_step(batch=batch, optimizer=optimizer)
    loss_history.append(output.total_loss)
    print(
        f"Step {step:02d} | total={output.total_loss:.6f} | "
        f"hard={output.hard_loss:.6f} | soft={output.soft_loss:.6f}"
    )

RuntimeError: cannot reshape tensor of 0 elements into shape [0, -1] because the unspecified dimension size -1 can be any value and is ambiguous

: 

## Quick sanity check

In [ ]:
print(f"Recorded {len(loss_history)} KD steps")
print(f"First loss: {loss_history[0]:.6f}")
print(f"Last loss:  {loss_history[-1]:.6f}")

This completes the first YOLO KD smoke flow. To move beyond smoke testing, replace synthetic batches with a real detection dataloader and include a detection-task hard loss (boxes + classes + DFL) alongside logits KD.